In [ ]:
from pathlib import Path

from matplotlib import pyplot as plt
from sbi.analysis import pairplot

from mach3sbitools.inference import InferenceHandler
from mach3sbitools.simulator import Simulator
from mach3sbitools.utils import MaCh3Logger, PosteriorConfig, TrainingConfig

MaCh3Logger("mach3sbitools")

In [ ]:
# Some global variables
BASE = Path("jupyter_tutorial")

DATA_FILE = BASE / "data/my_data.feather"
PRIOR_FILE = BASE / "prior/my_prior.pkl"
MODEL_FILE = BASE / "model/my_model.ckpt"
INFERENCE_FILE = BASE / "inference/my_inference.paraquet"

In [ ]:
# Firstly we load the simulator
simulator = Simulator(
    module_name="my_simulator",
    class_name="MySimulator",
    config=Path("physics_configs/PhysicsConfig.yaml"),
)

theta, x = simulator.simulate(100_000)
simulator.save(DATA_FILE, theta, x)

simulator.prior.save(PRIOR_FILE)

In [ ]:
# Now we setup training
posterior_config = PosteriorConfig(
    model="maf",
    hidden_features=64,
    num_transforms=5,
    dropout_probability=0.1,
)

training_config = TrainingConfig(
    save_path=MODEL_FILE,
    max_epochs=5000,
    show_progress=True,
    tensorboard_dir="tensor-logs",
)

trainer = InferenceHandler(PRIOR_FILE)
trainer.set_dataset(DATA_FILE.parent)
trainer.load_training_data()
trainer.create_posterior(posterior_config)

In [ ]:
# Now we run training
trainer.train_posterior(training_config, model_config=posterior_config)

In [ ]:
# data_bins = simulator.simulator_wrapper.get_data_bins()
data_bins = simulator.simulator_wrapper.get_data_bins()
samples = trainer.sample_posterior(1_000_000, data_bins)

In [ ]:
pairplot(samples, labels=simulator.simulator_wrapper.get_parameter_names())
plt.show()